In [1]:
!pip install torchvision streamlit streamlit-option-menu pyngrok diffusers transformers accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 107.2 MB/s eta 0:00:00


In [2]:
!ngrok config add-authtoken 3GtE58u1ax3OxBDsBtg0qWKNpoQ_89XRu4yAtvmTnbBPfadiR

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
!pip install fpdf2 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 15.8 MB/s eta 0:00:00


In [4]:
%%writefile app.py
import streamlit as st
from streamlit_option_menu import option_menu
from PIL import Image
import torch
from torchvision import models
from torchvision.models import MobileNet_V2_Weights
from diffusers import StableDiffusionImg2ImgPipeline
import sqlite3
import hashlib
import datetime
import io
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF

st.set_page_config(
    page_title="EcoCraft AI",
    page_icon="♻️",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ================================================================
# SESSION STATE
# ================================================================
defaults = {
    "logged_in": False, "user_name": "", "user_email": "",
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

# ================================================================
# DATABASE (SQLite - persists across logins in this runtime)
# ================================================================
@st.cache_resource
def get_db_connection():
    conn = sqlite3.connect("ecocraft.db", check_same_thread=False)
    cur = conn.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS users (
        email TEXT PRIMARY KEY, name TEXT, password_hash TEXT)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS history (
        id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, waste_category TEXT,
        detected_label TEXT, confidence REAL, eco_points INTEGER, timestamp TEXT)""")
    conn.commit()
    return conn

db_conn = get_db_connection()

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def register_user(name, email, password):
    cur = db_conn.cursor()
    cur.execute("SELECT email FROM users WHERE email = ?", (email,))
    if cur.fetchone() is not None:
        return False, "An account with this email already exists."
    cur.execute("INSERT INTO users (email, name, password_hash) VALUES (?, ?, ?)",
                (email, name, hash_password(password)))
    db_conn.commit()
    return True, "Registered successfully."

def verify_login(email, password):
    cur = db_conn.cursor()
    cur.execute("SELECT name, password_hash FROM users WHERE email = ?", (email,))
    row = cur.fetchone()
    if row is None:
        return False, None
    name, stored_hash = row
    return (True, name) if hash_password(password) == stored_hash else (False, None)

def log_analysis(email, waste_category, detected_label, confidence, eco_points):
    cur = db_conn.cursor()
    cur.execute("""INSERT INTO history (email, waste_category, detected_label, confidence, eco_points, timestamp)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                (email, waste_category, detected_label, confidence, eco_points,
                 datetime.datetime.now().isoformat()))
    db_conn.commit()

def get_user_history(email):
    return pd.read_sql_query(
        "SELECT waste_category, detected_label, confidence, eco_points, timestamp FROM history WHERE email = ? ORDER BY timestamp DESC",
        db_conn, params=(email,)
    )

def get_user_stats(email):
    df = get_user_history(email)
    if df.empty:
        return {"total_score": 0, "images_analyzed": 0, "category_counts": {}, "kg_diverted": 0.0, "df": df}
    return {
        "total_score": int(df["eco_points"].sum()),
        "images_analyzed": len(df),
        "category_counts": df["waste_category"].value_counts().to_dict(),
        "kg_diverted": round(len(df) * 0.2, 1),
        "df": df
    }

def already_did_challenge_today(email):
    df = get_user_history(email)
    if df.empty:
        return False
    today = datetime.date.today().isoformat()
    df["date"] = df["timestamp"].str[:10]
    return not df[(df["waste_category"] == "Daily Challenge") & (df["date"] == today)].empty

# ================================================================
# LOAD LOCAL MODELS (cached, load once)
# ================================================================
@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)
    model.eval()
    model.to(device)
    return model, weights.transforms(), weights.meta["categories"], device

model, preprocess, categories, device = load_model()

@st.cache_resource
def load_sd_pipeline():
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        safety_checker=None,
    )
    return pipe.to(device)

def generate_product_image(source_image, product_name, waste_category):
    sd_pipe = load_sd_pipeline()
    prompt = (f"a handmade {product_name} creatively upcycled from a {waste_category.lower()} item, "
              f"professional product photography, clean studio background, high detail, realistic")
    negative_prompt = "blurry, low quality, distorted, ugly, watermark, text"
    init_image = source_image.convert("RGB").resize((512, 512))
    result = sd_pipe(prompt=prompt, negative_prompt=negative_prompt, image=init_image,
                      strength=0.7, guidance_scale=7.5, num_inference_steps=30)
    return result.images[0]

# ================================================================
# CORE DATA
# ================================================================
WASTE_KEYWORDS = {
    "Plastic": ["bottle", "plastic bag", "pop bottle", "water jug", "container", "milk can"],
    "Metal": ["tin can", "can opener", "beer can", "spatula", "corkscrew", "lighter", "wok", "frying pan"],
    "Glass": ["wine bottle", "beer bottle", "vase", "goblet", "beer glass"],
    "Paper/Cardboard": ["carton", "cardboard", "envelope", "paper towel", "packet", "book jacket"],
    "Organic": ["banana", "orange", "apple", "corn", "mushroom", "vegetable", "fruit", "lemon"],
    "E-Waste": ["cellular telephone", "remote control", "modem", "hard disc", "laptop", "keyboard", "computer"],
    "Fabric": ["sweater", "jersey", "wool", "cloak", "poncho", "handkerchief"],
}

def classify_waste(label):
    label_lower = label.lower()
    for category, keywords in WASTE_KEYWORDS.items():
        if any(kw in label_lower for kw in keywords):
            return category
    return "General Waste"

PRODUCT_IDEAS = {
    "Plastic": {"products": ["Self-watering planter", "Bird feeder", "Pen/pencil holder", "Storage scoop"],
        "diy_steps": ["Rinse the bottle thoroughly and peel off any labels.",
            "Mark a cut line at your desired height.", "Cut carefully along the line.",
            "Sand the cut edge smooth.", "Poke drainage holes if making a planter.",
            "Decorate with paint or twine.", "Let paint dry fully before use."],
        "impact": "Plastic can take 400+ years to decompose. Reusing it keeps it out of waterways and reduces new plastic production."},
    "Metal": {"products": ["Pen holder", "Wind chime", "Planter", "Small storage tin"],
        "diy_steps": ["Wash the can thoroughly and remove the label.", "Sand down any sharp rim edges.",
            "Punch a hole near the top if making a wind chime.", "Paint the outside with acrylic paint.",
            "Let it dry completely, add a second coat if needed.", "Attach string or a hook."],
        "impact": "Recycling metal uses up to 95% less energy than producing new metal from raw ore."},
    "Glass": {"products": ["Candle holder", "Flower vase", "Drinking glass", "Terrarium"],
        "diy_steps": ["Soak in warm soapy water for 10-15 minutes.", "Scrub off the label and glue residue.",
            "Rinse and air dry completely.", "Decorate with twine or paint if desired.",
            "For a terrarium, layer stones, soil, then a small plant.", "Place in indirect sunlight."],
        "impact": "Glass is 100% recyclable and can be reused endlessly without quality loss."},
    "Paper/Cardboard": {"products": ["Organizer box", "Gift box", "Seed starter pots", "Wall art"],
        "diy_steps": ["Flatten and clean the cardboard.", "Measure and mark your desired shape.",
            "Cut along the marked lines.", "Fold along crease lines to form the box.",
            "Reinforce corners with tape or glue.", "Decorate or label as needed."],
        "impact": "Recycling paper saves trees, water, and energy compared to producing new paper."},
    "Organic": {"products": ["Compost", "Natural dye", "Plant fertilizer"],
        "diy_steps": ["Collect scraps (avoid meat/dairy for home compost).",
            "Mix 'green' scraps with 'brown' dry material.", "Layer scraps with soil in a bin.",
            "Turn the pile every few days.", "Keep it slightly moist, not soggy.",
            "After 4-6 weeks it becomes rich compost."],
        "impact": "Composting reduces methane emissions from landfills and creates nutrient-rich soil."},
    "E-Waste": {"products": ["Component art project", "Refurbish/donate", "Take apart for parts"],
        "diy_steps": ["Do NOT place in regular household trash.", "Check if it still works - consider donating.",
            "If not working, find a certified e-waste recycler.", "Remove personal data/batteries first.",
            "Drop off at the certified recycler."],
        "impact": "E-waste contains toxic materials like lead and mercury that can contaminate soil and water."},
    "Fabric": {"products": ["Tote bag", "Cleaning rags", "Patchwork quilt piece", "Pet bed stuffing"],
        "diy_steps": ["Wash and dry the fabric fully.", "Cut into your desired shape/size.",
            "Sew or hem raw edges to prevent fraying.", "For a tote bag, sew sides and attach handles.",
            "For rags, no sewing needed."],
        "impact": "Textile waste is one of the fastest-growing landfill categories; reuse reduces new textile demand."},
    "General Waste": {"products": ["Check local recycling guidelines"],
        "diy_steps": ["This item didn't confidently match a specific category.",
            "Check your local recycling center's guidelines.",
            "When unsure, dispose separately rather than contaminate a recycling batch."],
        "impact": "Proper sorting ensures recyclables don't get contaminated or sent to landfill unnecessarily."},
}

MATERIAL_INFO = {
    "Plastic": "Plastics are synthetic polymers, usually from petroleum - lightweight, moldable, and among the slowest materials to break down naturally.",
    "Metal": "Metals have a rigid, often reflective surface. Common recyclable metals like aluminum and steel can be melted down and reused indefinitely.",
    "Glass": "Glass is made from melted sand (silica) - transparent, brittle, non-porous, and infinitely recyclable with no quality loss.",
    "Paper/Cardboard": "Made from wood pulp fibers - fibrous, matte texture, biodegradable, though fibers weaken with repeated recycling.",
    "Organic": "Natural, biodegradable food/plant matter - ideal for composting rather than landfill disposal.",
    "E-Waste": "Electronics with plastic/metal composite housings, requiring specialized recycling due to hazardous internal components.",
    "Fabric": "Woven/knit fiber material. Natural fibers biodegrade slowly; synthetic fibers behave more like plastic.",
    "General Waste": "Didn't confidently match a specific material category based on visual features alone.",
}

ECO_POINTS = {"Plastic": 10, "Metal": 15, "Glass": 20, "Paper/Cardboard": 10,
              "Organic": 5, "E-Waste": 25, "Fabric": 10, "General Waste": 5, "Daily Challenge": 20}

IMPACT_FACTORS = {
    "Plastic":         {"co2": 0.08, "water": 3.0,  "energy": 0.10, "icon": "🥤"},
    "Metal":           {"co2": 0.15, "water": 1.5,  "energy": 0.60, "icon": "🥫"},
    "Glass":           {"co2": 0.20, "water": 0.5,  "energy": 0.30, "icon": "🍾"},
    "Paper/Cardboard": {"co2": 0.05, "water": 5.0,  "energy": 0.08, "icon": "📦"},
    "Organic":         {"co2": 0.03, "water": 0.2,  "energy": 0.02, "icon": "🍌"},
    "E-Waste":         {"co2": 1.20, "water": 8.0,  "energy": 2.50, "icon": "💻"},
    "Fabric":          {"co2": 0.25, "water": 12.0, "energy": 0.40, "icon": "👕"},
    "General Waste":   {"co2": 0.05, "water": 1.0,  "energy": 0.05, "icon": "🗑️"},
}

ACHIEVEMENTS = [
    (0, "🌱", "Eco Beginner"), (100, "🌿", "Green Warrior"), (300, "🌳", "Eco Hero"),
    (600, "🏅", "Planet Protector"), (1000, "🌍", "Climate Champion"),
]

def get_achievement(score):
    current = ACHIEVEMENTS[0]
    for threshold, icon, name in ACHIEVEMENTS:
        if score >= threshold:
            current = (threshold, icon, name)
    return current

def get_level(score):
    level = score // 100 + 1
    progress = score % 100
    return level, progress

SDG_INFO = {
    "SDG 11": ("Sustainable Cities and Communities", "Reducing waste and reusing materials supports cleaner, more livable cities."),
    "SDG 12": ("Responsible Consumption and Production", "Upcycling directly promotes reuse over disposal, cutting resource consumption."),
    "SDG 13": ("Climate Action", "Diverting waste from landfills lowers methane and CO₂ emissions."),
}

RECYCLING_CENTERS = [
    {"name": "GreenLoop Recycling Center", "distance": "1.8 km", "accepts": ["Plastic", "Metal", "Glass"], "hours": "9 AM - 6 PM", "phone": "+91-98765-11111"},
    {"name": "City Paper & Cardboard Hub", "distance": "2.4 km", "accepts": ["Paper/Cardboard"], "hours": "10 AM - 5 PM", "phone": "+91-98765-22222"},
    {"name": "E-Waste Safe Disposal Point", "distance": "3.1 km", "accepts": ["E-Waste"], "hours": "9 AM - 4 PM", "phone": "+91-98765-33333"},
    {"name": "Community Compost Yard", "distance": "1.2 km", "accepts": ["Organic"], "hours": "6 AM - 8 PM", "phone": "+91-98765-44444"},
    {"name": "Textile Reuse Collective", "distance": "4.0 km", "accepts": ["Fabric"], "hours": "11 AM - 7 PM", "phone": "+91-98765-55555"},
    {"name": "All-Materials Municipal Center", "distance": "5.5 km", "accepts": list(ECO_POINTS.keys()), "hours": "8 AM - 8 PM", "phone": "+91-98765-66666"},
]

DAILY_TIPS = [
    "Rinse containers before recycling - food residue can contaminate an entire batch.",
    "A reusable water bottle can save over 150 single-use bottles a year.",
    "Composting food scraps cuts methane emissions from landfills.",
    "Buying second-hand reduces the demand for new manufacturing and packaging.",
    "LED bulbs use up to 80% less energy than incandescent ones.",
    "Cardboard boxes can be reused 5-7 times before the fibers weaken too much.",
    "E-waste should never go in regular trash - it contains heavy metals.",
    "Air-drying clothes instead of using a dryer saves significant energy.",
    "A single tree can absorb about 21 kg of CO₂ per year.",
    "Cloth bags can replace over 1,000 plastic bags across their lifetime.",
]

DAILY_CHALLENGES = [
    "Recycle 5 plastic bottles today.", "Reuse a cardboard box instead of buying storage.",
    "Start a small compost bin for kitchen scraps.", "Donate one item of clothing instead of discarding it.",
    "Bring a reusable bag for your next shopping trip.", "Properly drop off one e-waste item at a recycler.",
    "Refill a glass jar instead of buying a new container.",
]

def eco_chat_response(question, category=None):
    q = question.lower()
    if "cost" in q or "price" in q or "expensive" in q:
        return "Most of these DIY upcycling projects cost close to nothing - you're mainly reusing materials you already have, plus maybe some paint, glue, or twine (usually under ₹50-100)."
    if "child" in q or "kid" in q:
        if category == "E-Waste":
            return "E-waste projects are NOT recommended for children - components can contain hazardous materials and sharp/small parts. Adult supervision or a certified recycler is best."
        if category == "Glass":
            return "Glass projects need adult supervision due to sharp edges, but can be a fun supervised activity for older children."
        return "Yes, most of these projects (plastic, cardboard, fabric) are safe for children with basic adult supervision, especially around cutting steps."
    if "safe" in q:
        return "Wash and dry the item fully before working with it, sand any sharp edges, and supervise any cutting steps - especially for glass or metal items."
    if "recycle" in q or "where" in q or "center" in q:
        return "Check the 'Recycling Centers' tab in the Eco Hub - it lists nearby centers along with what materials they accept."
    if "tool" in q or "need" in q:
        return "Common tools: scissors or a craft knife, sandpaper, paint/brushes, glue or tape, and gloves for safety. Specific projects may need a few extras noted in the DIY guide."
    if "make" in q or "how do i" in q or "diy" in q or "step" in q:
        if category and category in PRODUCT_IDEAS:
            steps = PRODUCT_IDEAS[category]["diy_steps"][:3]
            return "Here's a quick start: " + " ".join(f"({i+1}) {s}" for i, s in enumerate(steps)) + " See the full DIY guide in the Upload results for all steps."
        return "Upload a waste image first, then check the Upload tab's 'Step-by-Step DIY Guide' - I can give more specific answers once I know the material."
    return "I can help with questions about cost, safety, tools needed, recycling locations, or how to make the suggested product - try asking one of those!"

def analyze_image(image):
    img_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_idx = torch.topk(probs, 5)
    top_label = categories[top5_idx[0]]
    confidence = top5_prob[0].item() * 100
    waste_category = classify_waste(top_label)
    factors = IMPACT_FACTORS[waste_category]
    return {
        "detected_label": top_label, "confidence": confidence, "waste_category": waste_category,
        "info": PRODUCT_IDEAS[waste_category], "material_explanation": MATERIAL_INFO[waste_category],
        "eco_points": ECO_POINTS[waste_category],
        "co2_saved": factors["co2"], "water_saved": factors["water"], "energy_saved": factors["energy"],
        "trees_saved": round(factors["co2"] / 21, 4),
    }

def _pdf_safe(text):
    return text.encode("latin-1", "replace").decode("latin-1")

# ================================================================
# FIXED PDF BUILDER (Resets X position to prevent horizontal overflow)
# ================================================================
def build_pdf_report(user_name, result, image, generated_image=None, product_name=None):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_fill_color(22, 163, 74)
    pdf.rect(0, 0, 210, 25, "F")
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_xy(10, 7)
    pdf.cell(0, 10, "EcoCraft AI - Waste Analysis Report", ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.ln(15)

    pdf.set_font("Helvetica", "", 11)
    pdf.set_x(10)
    pdf.cell(0, 8, _pdf_safe(f"User: {user_name}"), ln=True)
    pdf.set_x(10)
    pdf.cell(0, 8, f"Date & Time: {datetime.datetime.now().strftime('%b %d, %Y - %I:%M %p')}", ln=True)
    pdf.ln(4)

    img_path = "/tmp/report_upload.png"
    image.convert("RGB").save(img_path)
    pdf.set_x(10)
    pdf.image(img_path, w=70)
    pdf.ln(4)

    pdf.set_font("Helvetica", "B", 13)
    pdf.set_text_color(22, 163, 74)
    pdf.set_x(10)
    pdf.cell(0, 8, "Detection Results", ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font("Helvetica", "", 11)
    pdf.set_x(10)
    pdf.multi_cell(0, 7, _pdf_safe(f"Waste Category: {result['waste_category']}\n"
                          f"Detected Material: {result['detected_label']}\n"
                          f"Confidence: {result['confidence']:.1f}%\n"
                          f"Eco Score Earned: {result['eco_points']} points"))
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.set_text_color(22, 163, 74)
    pdf.set_x(10)
    pdf.cell(0, 8, "Environmental Savings (estimated)", ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font("Helvetica", "", 11)
    pdf.set_x(10)
    pdf.multi_cell(0, 7, f"CO2 Saved: {result['co2_saved']} kg\n"
                          f"Water Saved: {result['water_saved']} L\n"
                          f"Energy Saved: {result['energy_saved']} kWh\n"
                          f"Equivalent Trees Saved: {result['trees_saved']}")
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.set_text_color(22, 163, 74)
    pdf.set_x(10)
    pdf.cell(0, 8, "DIY Guide", ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font("Helvetica", "", 11)
    for i, step in enumerate(result["info"]["diy_steps"], 1):
        pdf.set_x(10)
        pdf.multi_cell(0, 7, _pdf_safe(f"{i}. {step}"))
    pdf.ln(2)

    pdf.set_font("Helvetica", "B", 13)
    pdf.set_text_color(22, 163, 74)
    pdf.set_x(10)
    pdf.cell(0, 8, "Environmental Impact", ln=True)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font("Helvetica", "", 11)
    pdf.set_x(10)
    pdf.multi_cell(0, 7, _pdf_safe(result["info"]["impact"]))

    if generated_image is not None:
        pdf.ln(2)
        pdf.set_font("Helvetica", "B", 13)
        pdf.set_text_color(22, 163, 74)
        pdf.set_x(10)
        pdf.cell(0, 8, _pdf_safe(f"Generated Product: {product_name}"), ln=True)
        pdf.set_text_color(0, 0, 0)
        gen_path = "/tmp/report_generated.png"
        generated_image.convert("RGB").save(gen_path)
        pdf.set_x(10)
        pdf.image(gen_path, w=70)

    pdf.output("/tmp/ecocraft_report.pdf")
    with open("/tmp/ecocraft_report.pdf", "rb") as f:
        return f.read()

# ================================================================
# STYLE
# ================================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700;800&display=swap');
html,body,[class*="css"]{ font-family:'Poppins',sans-serif; }
#MainMenu{visibility:hidden;} footer{visibility:hidden;} header{visibility:hidden;}
.stApp{ background:linear-gradient(-45deg,#050505,#0a0f0a,#050505,#0a1410); background-size:400% 400%;
    animation:gradientShift 22s ease infinite; color:white; }
@keyframes gradientShift{ 0%{background-position:0% 50%;} 50%{background-position:100% 50%;} 100%{background-position:0% 50%;} }
section.main>div{ padding-top:1rem; }

::-webkit-scrollbar{ width:10px; }
::-webkit-scrollbar-track{ background:#0b0b0b; }
::-webkit-scrollbar-thumb{ background:linear-gradient(#22C55E,#16A34A); border-radius:10px; }

.hero{ background:linear-gradient(145deg,#0b0b0b,#171717); border:1px solid rgba(34,197,94,.25);
    border-radius:30px; padding:65px; box-shadow:0px 15px 45px rgba(0,0,0,.45); }
.hero-title{ font-size:70px; font-weight:800; animation:fadeInUp 1s ease-out, shimmerText 6s linear infinite;
    background:linear-gradient(90deg,#ffffff,#22C55E,#ffffff); background-size:200% auto;
    -webkit-background-clip:text; background-clip:text; -webkit-text-fill-color:transparent; }
@keyframes shimmerText{ 0%{background-position:0% center;} 100%{background-position:200% center;} }
.hero-sub{ font-size:28px; color:#22C55E; margin-top:20px; margin-bottom:25px; animation:fadeInUp 1.2s ease-out; }
.hero p{ color:#D1D5DB; font-size:18px; line-height:1.8; animation:fadeInUp 1.4s ease-out; }

div[data-testid="stHorizontalBlock"] a{ transition:.3s !important; }
.nav-link:hover{ background-color:rgba(34,197,94,.15) !important; color:#22C55E !important; }
.nav-link-selected{ box-shadow:0px 0px 18px rgba(34,197,94,.55) !important; }

.earth-spin{ font-size:140px; display:inline-block; animation:spinEarth 18s linear infinite;
    filter:drop-shadow(0px 0px 25px rgba(34,197,94,.5)); }
@keyframes spinEarth{ from{transform:rotate(0deg);} to{transform:rotate(360deg);} }
@keyframes fadeInUp{ from{opacity:0; transform:translateY(25px);} to{opacity:1; transform:translateY(0);} }
@keyframes fadeIn{ from{opacity:0;} to{opacity:1;} }
@keyframes pulseGlow{ 0%{box-shadow:0 0 0px rgba(34,197,94,.4);} 50%{box-shadow:0 0 35px rgba(34,197,94,.55);} 100%{box-shadow:0 0 0px rgba(34,197,94,.4);} }

.leaf{ position:fixed; top:-40px; font-size:22px; opacity:.55; animation:leafFall linear infinite; pointer-events:none; z-index:0; }
@keyframes leafFall{ 0%{transform:translateY(0) rotate(0deg); opacity:0;} 10%{opacity:.55;} 90%{opacity:.4;}
    100%{transform:translateY(105vh) rotate(360deg); opacity:0;} }

.glass-card{ background:rgba(255,255,255,0.035); backdrop-filter:blur(12px); border:1px solid rgba(34,197,94,.25);
    border-radius:22px; padding:28px; transition:.35s; animation:fadeIn .8s ease-out; }
.glass-card:hover{ border:1px solid #22C55E; transform:translateY(-4px); box-shadow:0px 0px 25px rgba(34,197,94,.25); }

.feature-card{ background:#111111; border-radius:25px; padding:35px; text-align:center; border:1px solid #222;
    transition:.4s; height:280px; }
.feature-card:hover{ transform:translateY(-10px); border:1px solid #22C55E; box-shadow:0px 0px 30px rgba(34,197,94,.35); }
.feature-icon{ font-size:55px; margin-bottom:15px; }

.dash-header{ background:linear-gradient(145deg,#0b0b0b,#171717); border:1px solid rgba(34,197,94,.25);
    border-radius:25px; padding:40px; margin-bottom:30px; box-shadow:0px 10px 30px rgba(0,0,0,.4); }

.stat-card{ background:#111111; border:1px solid #222; border-radius:20px; padding:28px; text-align:center; transition:.35s; }
.stat-card:hover{ border:1px solid #22C55E; transform:translateY(-6px); box-shadow:0px 0px 25px rgba(34,197,94,.25); }
.stat-number{ font-size:32px; font-weight:700; color:#22C55E; }

.result-card{ background:#111111; border:1px solid rgba(34,197,94,.3); border-radius:25px; padding:35px;
    margin-top:20px; animation:fadeIn .8s ease-out, pulseGlow 2.5s ease-in-out 1; }
.category-badge{ display:inline-block; background:linear-gradient(90deg,#22C55E,#16A34A); color:white;
    padding:8px 20px; border-radius:30px; font-weight:600; font-size:15px; margin-right:8px; }
.product-chip{ display:inline-block; background:#1a1a1a; border:1px solid #22C55E; color:#22C55E;
    padding:8px 16px; border-radius:20px; margin:5px; font-size:14px; }

.badge-card{ background:linear-gradient(145deg,#0b0b0b,#171717); border:1px solid rgba(34,197,94,.3);
    border-radius:20px; padding:22px; text-align:center; transition:.3s; }
.badge-card.active{ border:1px solid #22C55E; box-shadow:0px 0px 20px rgba(34,197,94,.35); animation:pulseGlow 3s ease-in-out infinite; }
.badge-card:hover{ transform:translateY(-5px); }
.sdg-card{ background:#111111; border-left:4px solid #22C55E; border-radius:12px; padding:18px 22px; margin-bottom:12px;
    transition:.3s; }
.sdg-card:hover{ background:#161616; transform:translateX(4px); }

.stButton>button{ width:100%; background:linear-gradient(90deg,#22C55E,#16A34A); color:white; border:none;
    border-radius:40px; font-size:18px; font-weight:600; padding:15px; transition:.3s; }
.stButton>button:hover{ transform:scale(1.03); box-shadow:0px 0px 20px rgba(34,197,94,.5); }
.stButton>button:active{ transform:scale(.98); }
input{ border-radius:15px!important; transition:.3s; }
input:focus{ box-shadow:0px 0px 12px rgba(34,197,94,.5) !important; border-color:#22C55E !important; }

div[data-testid="stProgress"] > div > div{ background:linear-gradient(90deg,#22C55E,#16A34A) !important;
    border-radius:20px; }
div[data-testid="stProgress"]{ border-radius:20px; overflow:hidden; }

.product-chip{ display:inline-block; background:#1a1a1a; border:1px solid #22C55E; color:#22C55E;
    padding:8px 16px; border-radius:20px; margin:5px; font-size:14px; transition:.25s; }
.product-chip:hover{ background:#22C55E; color:#0b0b0b; transform:translateY(-2px); }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="leaf" style="left:15%; animation-duration:14s; animation-delay:0s;">🍃</div>
<div class="leaf" style="left:40%; animation-duration:19s; animation-delay:3s; font-size:16px;">🍂</div>
<div class="leaf" style="left:65%; animation-duration:16s; animation-delay:6s;">🍃</div>
<div class="leaf" style="left:85%; animation-duration:21s; animation-delay:2s; font-size:18px;">🍂</div>
""", unsafe_allow_html=True)

# ================================================================
# NAV
# ================================================================
if st.session_state.logged_in:
    menu_options = ["Dashboard", "Upload", "Eco Hub", "Logout"]
    menu_icons = ["speedometer2", "cloud-upload", "compass", "box-arrow-right"]
else:
    menu_options = ["Home", "Login", "Register"]
    menu_icons = ["house", "box-arrow-in-right", "person-plus"]

selected = option_menu(menu_title=None, options=menu_options, icons=menu_icons,
                        orientation="horizontal", default_index=0,
                        styles={"nav-link-selected": {"background-color": "#22C55E"}})

# ================================================================
# HOME
# ================================================================
if selected == "Home":
    left, right = st.columns([1.3, 1], gap="large")
    with left:
        st.markdown("""<div class="hero"><div class="hero-title">♻️ EcoCraft AI</div>
        <div class="hero-sub">Transforming Waste into Valuable Products</div>
        <p>Turn everyday waste into useful products with the power of Artificial Intelligence.
        Upload an image and receive smart recycling guidance, creative upcycling ideas, and
        eco-friendly suggestions in seconds.</p></div>""", unsafe_allow_html=True)
        st.write("")
        if st.button("🚀 Get Started"):
            st.success("Head to Login or Register above to begin!")
    with right:
        st.markdown("""<div style="background:#111111; border-radius:30px; padding:40px;
        border:1px solid rgba(34,197,94,.3); text-align:center; box-shadow:0px 0px 20px rgba(34,197,94,.2);">
        <h1 class="earth-spin">♻️</h1><h2 style="color:#22C55E;">Eco-Friendly AI</h2>
        <p style="color:white;">Building a Sustainable Future</p></div>""", unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)
    with c1:
        st.markdown("""<div class="feature-card"><div class="feature-icon">📷</div><h3>Smart Detection</h3>
        <p>Upload any waste image and let AI identify the waste type instantly.</p></div>""", unsafe_allow_html=True)
    with c2:
        st.markdown("""<div class="feature-card"><div class="feature-icon">🤖</div><h3>AI Generated Ideas</h3>
        <p>Receive creative upcycling ideas and a visualized transformed product.</p></div>""", unsafe_allow_html=True)
    with c3:
        st.markdown("""<div class="feature-card"><div class="feature-icon">🌱</div><h3>Eco Impact</h3>
        <p>Track your carbon, water, and energy savings with every item you reuse.</p></div>""", unsafe_allow_html=True)

# ================================================================
# LOGIN
# ================================================================
elif selected == "Login":
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("""<div style="background:#111111; padding:40px; border-radius:25px;
        border:1px solid rgba(34,197,94,.3); box-shadow:0 0 25px rgba(34,197,94,.15);">
        <center><h1 style="color:white;">🌱 Welcome Back</h1>
        <p style="color:#BDBDBD;">Login to continue using EcoCraft AI</p></center></div>""", unsafe_allow_html=True)
        st.write("")
        email = st.text_input("📧 Email")
        password = st.text_input("🔒 Password", type="password")
        st.write("")
        if st.button("🚀 Login", use_container_width=True):
            if email == "" or password == "":
                st.error("Please fill all fields.")
            else:
                ok, name = verify_login(email, password)
                if ok:
                    st.session_state.logged_in = True
                    st.session_state.user_name = name
                    st.session_state.user_email = email
                    st.success("Login Successful ✅ Redirecting to Dashboard...")
                    st.rerun()
                else:
                    st.error("Invalid email or password.")
        st.divider()
        st.info("Don't have an account? Go to Register.")

# ================================================================
# REGISTER
# ================================================================
elif selected == "Register":
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("""<div style="background:#111111; padding:40px; border-radius:25px;
        border:1px solid rgba(34,197,94,.3); box-shadow:0 0 25px rgba(34,197,94,.15);">
        <center><h1 style="color:white;">🌿 Create Account</h1>
        <p style="color:#BDBDBD;">Join EcoCraft AI Today</p></center></div>""", unsafe_allow_html=True)
        st.write("")
        name = st.text_input("👤 Full Name")
        email = st.text_input("📧 Email Address")
        password = st.text_input("🔒 Password", type="password")
        confirm = st.text_input("🔑 Confirm Password", type="password")
        agree = st.checkbox("I agree to the Terms & Conditions")
        st.write("")
        if st.button("📝 Create Account", use_container_width=True):
            if not all([name, email, password, confirm]):
                st.error("Please fill all fields.")
            elif password != confirm:
                st.error("Passwords do not match.")
            elif not agree:
                st.warning("Please accept the Terms & Conditions.")
            else:
                ok, msg = register_user(name, email, password)
                if ok:
                    st.session_state.logged_in = True
                    st.session_state.user_name = name
                    st.session_state.user_email = email
                    st.success("Registration Successful 🎉 Redirecting to Dashboard...")
                    st.rerun()
                else:
                    st.error(msg)
        st.divider()
        st.info("Already have an account? Login from the menu.")

# ================================================================
# DASHBOARD
# ================================================================
elif selected == "Dashboard":
    stats = get_user_stats(st.session_state.user_email)
    level, level_progress = get_level(stats["total_score"])
    _, ach_icon, ach_name = get_achievement(stats["total_score"])
    tip = DAILY_TIPS[datetime.date.today().toordinal() % len(DAILY_TIPS)]

    st.markdown(f"""<div class="dash-header"><h1 style="color:white;">🌿 Welcome back, {st.session_state.user_name.title()}!</h1>
    <p style="color:#BDBDBD;font-size:17px;">Level {level} {ach_icon} {ach_name} - keep going, {100-level_progress} points to next level.</p>
    </div>""", unsafe_allow_html=True)
    st.progress(level_progress / 100)

    s1, s2, s3 = st.columns(3)
    with s1:
        st.markdown(f"""<div class="stat-card"><div class="stat-number">{stats['images_analyzed']}</div>
        <p style="color:#D1D5DB;">Images Analyzed</p></div>""", unsafe_allow_html=True)
    with s2:
        st.markdown(f"""<div class="stat-card"><div class="stat-number">{stats['total_score']}</div>
        <p style="color:#D1D5DB;">🌟 Total Eco Score</p></div>""", unsafe_allow_html=True)
    with s3:
        st.markdown(f"""<div class="stat-card"><div class="stat-number">{stats['kg_diverted']} kg</div>
        <p style="color:#D1D5DB;">Waste Diverted (est.)</p></div>""", unsafe_allow_html=True)

    st.write("")
    st.markdown(f"""<div class="glass-card">🌿 <b>Today's Eco Tip:</b> {tip}</div>""", unsafe_allow_html=True)
    st.write("")

    if stats["images_analyzed"] > 0:
        st.markdown("### 📊 Your Analytics")
        colA, colB = st.columns(2)
        with colA:
            fig, ax = plt.subplots(figsize=(4, 3.2), facecolor="#0b0b0b")
            ax.set_facecolor("#0b0b0b")
            cats = list(stats["category_counts"].keys())
            counts = list(stats["category_counts"].values())
            ax.pie(counts, labels=cats, autopct="%1.0f%%", textprops={"color": "white", "fontsize": 8},
                   colors=plt.cm.Greens_r([i / max(len(cats), 1) for i in range(len(cats))]))
            ax.set_title("Category Breakdown", color="white")
            st.pyplot(fig)
        with colB:
            df = stats["df"].copy()
            df["date"] = pd.to_datetime(df["timestamp"]).dt.date
            daily = df.groupby("date")["eco_points"].sum().tail(7)
            fig2, ax2 = plt.subplots(figsize=(4, 3.2), facecolor="#0b0b0b")
            ax2.set_facecolor("#0b0b0b")
            ax2.bar(daily.index.astype(str), daily.values, color="#22C55E")
            ax2.set_title("Eco Points - Last 7 Days", color="white")
            ax2.tick_params(colors="white", labelsize=7)
            for spine in ax2.spines.values():
                spine.set_color("#333")
            st.pyplot(fig2)
    else:
        st.info("👉 Head to the Upload tab to analyze your first waste image and start building your stats.")

    st.write("")
    st.markdown("### 🌱 Daily Eco Challenge")
    challenge = DAILY_CHALLENGES[datetime.date.today().weekday()]
    done_today = already_did_challenge_today(st.session_state.user_email)
    st.markdown(f"""<div class="glass-card">{challenge}</div>""", unsafe_allow_html=True)
    if done_today:
        st.success("✅ Completed today! Come back tomorrow for a new challenge.")
    else:
        if st.button("✅ Mark Challenge Complete (+20 Eco Points)"):
            log_analysis(st.session_state.user_email, "Daily Challenge", challenge, 100.0, 20)
            st.success("Great work! +20 Eco Points added.")
            st.rerun()

# ================================================================
# UPLOAD
# ================================================================
elif selected == "Upload":
    st.markdown("""<h1 style="text-align:center;color:white;">📷 Upload Waste Image</h1>""", unsafe_allow_html=True)
    st.caption(f"⚙️ Running on: {'NVIDIA GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
    st.write("")

    uploaded_file = st.file_uploader("Choose an image", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        image = Image.open(uploaded_file).convert("RGB")
        st.image(image, caption="Uploaded Image", use_container_width=True)
        st.write("")

        if st.button("🤖 Analyze Waste"):
            with st.spinner("Analyzing locally (no API used)..."):
                st.session_state.analysis_result = analyze_image(image)
                st.session_state.analysis_image = image
                r = st.session_state.analysis_result
                log_analysis(st.session_state.user_email, r["waste_category"], r["detected_label"],
                             r["confidence"], r["eco_points"])

        if "analysis_result" in st.session_state:
            result = st.session_state.analysis_result
            image = st.session_state.analysis_image

            st.markdown(f"""<div class="result-card"><span class="category-badge">{result['waste_category']}</span>
            <span class="category-badge" style="background:linear-gradient(90deg,#16A34A,#0F7A38);">
            🌟 +{result['eco_points']} Eco Points</span>
            <p style="color:#BDBDBD;margin-top:15px;">Detected: <b style="color:white;">{result['detected_label']}</b>
            ({result['confidence']:.1f}% confidence)</p></div>""", unsafe_allow_html=True)

            st.write("")
            st.markdown("### 🧩 AI Material Explanation")
            st.markdown(f"<p style='color:#D1D5DB;'>{result['material_explanation']}</p>", unsafe_allow_html=True)

            st.write("")
            st.markdown("### 🌍 Carbon Footprint Calculator")
            fc1, fc2, fc3, fc4 = st.columns(4)
            fc1.metric("CO₂ Saved", f"{result['co2_saved']} kg")
            fc2.metric("Water Saved", f"{result['water_saved']} L")
            fc3.metric("Energy Saved", f"{result['energy_saved']} kWh")
            fc4.metric("Trees Equivalent", f"{result['trees_saved']}")

            st.write("")
            st.markdown("### 🛠️ Product Ideas")
            chips = "".join([f'<span class="product-chip">{p}</span>' for p in result['info']['products']])
            st.markdown(chips, unsafe_allow_html=True)

            st.write("")
            st.markdown("### 📋 Step-by-Step DIY Guide")
            steps_html = "".join([f"<li style='color:#D1D5DB;margin-bottom:8px;'>{s}</li>" for s in result['info']['diy_steps']])
            st.markdown(f"<ol style='padding-left:20px;'>{steps_html}</ol>", unsafe_allow_html=True)

            st.write("")
            st.markdown("### 🌍 Environmental Impact")
            st.markdown(f"<p style='color:#D1D5DB;'>{result['info']['impact']}</p>", unsafe_allow_html=True)

            st.write("")
            st.markdown("### 🎨 Visualize the Product (Before vs After)")
            chosen_product = st.selectbox("Pick a product idea to generate an image for:",
                                           result['info']['products'], key="chosen_product_select")
            if st.button("✨ Generate Product Image"):
                with st.spinner("Generating with Stable Diffusion on GPU (first run downloads ~4GB)..."):
                    st.session_state.generated_image = generate_product_image(image, chosen_product, result['waste_category'])
                    st.session_state.generated_product_name = chosen_product

            if "generated_image" in st.session_state:
                gcol1, gcol2 = st.columns(2)
                with gcol1:
                    st.image(image, caption="Original Upload (Before)", use_container_width=True)
                with gcol2:
                    st.image(st.session_state.generated_image,
                              caption=f"Generated: {st.session_state.generated_product_name} (After)",
                              use_container_width=True)
                buf = io.BytesIO()
                st.session_state.generated_image.save(buf, format="PNG")
                st.download_button("📥 Download Generated Image", data=buf.getvalue(),
                                    file_name="ecocraft_generated_product.png", mime="image/png")

            st.write("")
            st.markdown("### 📄 Download Full Report")
            pdf_bytes = build_pdf_report(
                st.session_state.user_name, result, image,
                generated_image=st.session_state.get("generated_image"),
                product_name=st.session_state.get("generated_product_name")
            )
            st.download_button("📄 Download PDF Report", data=pdf_bytes,
                                file_name="EcoCraft_AI_Report.pdf", mime="application/pdf")

# ================================================================
# ECO HUB (Chatbot, Recycling Centers, Achievements, SDGs)
# ================================================================
elif selected == "Eco Hub":
    st.markdown("""<h1 style="text-align:center;color:white;">🌍 Eco Hub</h1>""", unsafe_allow_html=True)
    tabs = st.tabs(["🤖 Eco Assistant", "🗺️ Recycling Centers", "🏅 Achievements", "🌐 SDGs"])

    with tabs[0]:
        st.markdown("Ask about DIY steps, cost, safety, tools, or where to recycle.")
        last_category = st.session_state.get("analysis_result", {}).get("waste_category") if "analysis_result" in st.session_state else None
        if "chat_history" not in st.session_state:
            st.session_state.chat_history = []
        user_q = st.text_input("Your question", key="chat_input")
        if st.button("Ask Eco Assistant"):
            if user_q.strip():
                answer = eco_chat_response(user_q, last_category)
                st.session_state.chat_history.append((user_q, answer))
        for q, a in reversed(st.session_state.chat_history):
            st.markdown(f"""<div class="glass-card" style="margin-bottom:12px;">
            <b style="color:#22C55E;">You:</b> {q}<br><b style="color:#22C55E;">Eco Assistant:</b> {a}</div>""",
            unsafe_allow_html=True)

    with tabs[1]:
        st.caption("Sample data shown - connect a live maps service for real-time results.")
        for center in RECYCLING_CENTERS:
            st.markdown(f"""<div class="glass-card" style="margin-bottom:14px;">
            <h4 style="color:white;margin:0;">{center['name']}</h4>
            <p style="color:#BDBDBD;margin:6px 0;">📍 {center['distance']} away &nbsp;|&nbsp; 🕒 {center['hours']} &nbsp;|&nbsp; 📞 {center['phone']}</p>
            <p style="color:#22C55E;margin:0;">Accepts: {', '.join(center['accepts'])}</p>
            </div>""", unsafe_allow_html=True)

    with tabs[2]:
        stats = get_user_stats(st.session_state.user_email)
        score = stats["total_score"]
        st.markdown(f"### Your Eco Score: {score} points")
        cols = st.columns(len(ACHIEVEMENTS))
        for i, (threshold, icon, name) in enumerate(ACHIEVEMENTS):
            active = "active" if score >= threshold else ""
            with cols[i]:
                st.markdown(f"""<div class="badge-card {active}"><div style="font-size:40px;">{icon}</div>
                <b style="color:white;">{name}</b><br><span style="color:#888;font-size:12px;">{threshold}+ pts</span></div>""",
                unsafe_allow_html=True)

    with tabs[3]:
        active_category = st.session_state.get("analysis_result", {}).get("waste_category") if "analysis_result" in st.session_state else None
        for sdg, (title, desc) in SDG_INFO.items():
            st.markdown(f"""<div class="sdg-card"><b style="color:#22C55E;">{sdg}: {title}</b>
            <p style="color:#D1D5DB;margin:6px 0 0 0;">{desc}</p></div>""", unsafe_allow_html=True)
        if active_category:
            st.info(f"Your last detected item ({active_category}) directly supports these goals by being reused instead of landfilled.")

# ================================================================
# LOGOUT
# ================================================================
elif selected == "Logout":
    st.session_state.logged_in = False
    st.session_state.user_name = ""
    st.session_state.user_email = ""
    st.success("You've been logged out.")
    st.rerun()

Writing app.py


In [5]:
!pip install pyngrok -q

In [30]:
!pip install torchvision streamlit streamlit-option-menu pyngrok diffusers transformers accelerate -q

In [6]:
from pyngrok import ngrok
ngrok.kill()

In [32]:
!pip install fpdf2 -q

In [7]:
import threading, os, time

def run():
    os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run)

thread.start()

time.sleep(8)

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://handset-whoops-aloe.ngrok-free.dev" -> "http://localhost:8501"
